# Mini-TP 4 — Puntúa tu modelo sobre un flujo (Sesión 4)

Aplico **mi modelo** (`iris-rf`, el mismo servido por REST en el Mini-TP 1,
descripto por GraphQL en el Mini-TP 2 y expuesto por gRPC en el Mini-TP 3)
sobre un **flujo de eventos** en lugar de pedidos sueltos.

El pipeline vive en `src/model_service/streaming.py` (compartido, como el
resto): simula un flujo con las 4 features del modelo, corre inferencia
**online** evento a evento, calcula métricas por **ventana deslizante**
(throughput, latencia p95, un indicador de **drift** de entradas) y dispara
una **alerta** cuando la ventana se aleja demasiado de lo normal. Al final lo
comparo contra puntuar el mismo lote en **batch**.


## 1. El flujo — features + drift inyectado

`generar_eventos` arma `n` eventos con las 4 features de `iris-rf`, muestreadas
alrededor de la media/desvío real del dataset de entrenamiento
(`sklearn.datasets.load_iris`). A partir de la fracción `drift_at` del flujo
corro esa distribución **+2.5 sigma** por feature — simula un sensor o una
población de entrada distinta a la que vio el modelo en entrenamiento — para
poder disparar el detector de drift más abajo.

In [1]:
from model_service.streaming import (
    BASELINE_MEAN, BASELINE_STD, generar_eventos, correr_streaming, puntuar_batch,
)
from model_service.model import FEATURES

N = 800
eventos = generar_eventos(N, drift_at=0.5)

print("features       :", FEATURES)
print("baseline mean  :", BASELINE_MEAN.round(2))
print("baseline std   :", BASELINE_STD.round(2))
print(f"{N} eventos generados; drift a partir del evento {N // 2}")


features       : ['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']
baseline mean  : [5.84 3.06 3.76 1.2 ]
baseline std   : [0.83 0.43 1.76 0.76]
800 eventos generados; drift a partir del evento 400


## 2. Consumidor: inferencia online + métricas por ventana

`correr_streaming` levanta un **productor** (encola los eventos, como si
vinieran de un topic Kafka) y un **consumidor** en threads separados. El
consumidor llama a `model.predict` **por cada evento** (inferencia online) y
mantiene una ventana deslizante de los últimos 100 eventos. Cada 50 eventos
recalcula, por feature, cuántos desvíos estándar (`z-score`) se alejó la media
de la ventana respecto de la media de entrenamiento; si algún feature supera
`DRIFT_Z_THRESHOLD` (2.5 sigma), registra una alerta.

In [2]:
resultado = correr_streaming(eventos)

print(f"throughput       : {resultado.throughput_ev_s:.0f} ev/s")
print(f"latencia p95     : {resultado.p95_ms:.3f} ms")
print(f"eventos procesados: {len(resultado.latencias_ms)}")
print(f"alertas de drift  : {len(resultado.alertas)}")
for a in resultado.alertas:
    print("  ", a)


throughput       : 77 ev/s
latencia p95     : 14.389 ms
eventos procesados: 800
alertas de drift  : 7
   {'evento': 500, 'feature': 'sepal_width_cm', 'z_score': 2.57}
   {'evento': 550, 'feature': 'petal_width_cm', 'z_score': 2.62}
   {'evento': 600, 'feature': 'petal_width_cm', 'z_score': 2.75}
   {'evento': 650, 'feature': 'petal_width_cm', 'z_score': 2.54}
   {'evento': 700, 'feature': 'sepal_length_cm', 'z_score': 2.58}
   {'evento': 750, 'feature': 'petal_length_cm', 'z_score': 2.71}
   {'evento': 800, 'feature': 'petal_length_cm', 'z_score': 2.69}


**Lectura:** las alertas empiezan a aparecer poco después del evento 400
(donde inyecté el drift) — la primera ventana de 100 eventos que queda
*enteramente* del lado corrido del flujo dispara la alarma en cuanto se
revisa (cada 50 eventos). El feature que más se dispara varía de chequeo a
chequeo porque los cuatro se corrieron a la vez y el ruido gaussiano hace que
alguno supere el umbral primero.

## 3. Comparación contra batch

Puntúo el **mismo** conjunto de eventos de una sola vez con
`model.predict_batch` (un solo llamado a `pipeline.predict` sobre toda la
matriz) y comparo el costo por evento contra la inferencia online del
streaming.

In [3]:
preds_batch, batch_ms_total = puntuar_batch(eventos)
batch_ms_evento = batch_ms_total / len(eventos)
online_ms_evento = sum(resultado.latencias_ms) / len(resultado.latencias_ms)

print(f"batch : {batch_ms_total:.3f} ms totales -> {batch_ms_evento:.4f} ms/evento")
print(f"online: {online_ms_evento:.4f} ms/evento (latencia p95 = {resultado.p95_ms:.3f} ms)")
print(f"batch ~{online_ms_evento / batch_ms_evento:.0f}x más barato por evento que online")

coinciden = sum(1 for a, b in zip(preds_batch, resultado.predicciones) if a == b)
print(f"predicciones coinciden batch vs online: {coinciden}/{len(eventos)}")


batch : 17.699 ms totales -> 0.0221 ms/evento
online: 13.0635 ms/evento (latencia p95 = 14.389 ms)
batch ~590x más barato por evento que online
predicciones coinciden batch vs online: 800/800


### Reflexión — streaming vs batch

- **Qué me dio el streaming que el batch no:** resultado **por evento**, en
  el momento en que llega — puedo actuar sobre la predicción del evento 1 sin
  esperar a que lleguen los otros 799 — y una **señal de drift en vivo**: la
  alerta se dispara a los pocos eventos de que cambió la distribución de
  entrada, no recién cuando alguien decide re-evaluar el modelo en batch días
  después.
- **Costo de esa inmediatez:** el batch resultó muchísimo más barato por
  evento (una sola llamada vectorizada a `pipeline.predict` vs. N llamadas
  individuales con overhead de cola/hilo en el medio) — la latencia p95 del
  streaming está dominada por ese overhead por-evento, no por el modelo en
  sí (que es el mismo `RandomForestClassifier` de 200 árboles del TP1/TP3).
- **Cuándo usaría cada uno:** streaming para *scoring* donde la frescura
  importa (fraude, recomendación en el momento, monitoreo de sensores) o
  donde necesito la señal de drift temprano; batch para reportes/reentreno
  periódico o cualquier caso donde puedo esperar una ventana y prefiero el
  throughput y el costo por predicción del procesamiento vectorizado.
- **Cómo detecté el drift:** un `z-score` de la media de la ventana deslizante
  contra la media/desvío de entrenamiento, por feature — simple e interpretable
  (“la ventana está a X sigma de lo normal”), aunque sensible al tamaño de
  ventana y solo mira la media (no detecta corrimientos de varianza o forma).
  Ante una alerta en producción: primero **avisar** (no autocorregir solo),
  revisar si es un problema de datos upstream (sensor roto, cambio de
  esquema) antes de asumir que el modelo quedó obsoleto, y si el drift
  persiste y coincide con caída de métricas de negocio, **reentrenar** con
  datos recientes; si la señal es ambigua, **degradar** a una regla más
  simple o pedir revisión humana en vez de servir predicciones de un modelo
  que ya no ve el mismo tipo de entrada que en entrenamiento.


## 4. (Extensión posible) Con Kafka en vez de la cola en memoria

El productor/consumidor de `streaming.py` usa una `queue.Queue` en memoria
para que el notebook corra sin infraestructura extra, pero la interfaz es la
misma que tendría un topic real: `productor` sólo necesita algo con `.put`, y
`consumidor` algo con `.get`. Reemplazar la cola por un `KafkaConsumer`
(`kafka-python`) apuntando a un topic `eventos` en un broker Redpanda/Kafka
local no cambia `model.predict` ni la lógica de ventana/drift — sólo el
transporte.